# COS 711 – Assignment 1
## Neural Network Fundamentals: Learning-Rate Warm-up and Pruning Robustness

**Task:** binary classification of pulsar candidates (HTRU2, 17 898 rows × 8 features, only ≈ 9 % positives) with a multilayer perceptron (MLP).

| Section | Brief | Content |
|---|---|---|
| 0 | – | Setup, configuration, reproducibility |
| 1 | §2.1 | Load data and create the train / validation / test split |
| 2 | §2.1 (1–5) | **Data preparation**: feature relevance, outliers, scaling, splitting, class imbalance |
| 3 | §2.2 | MLP + training utilities |
| 4 | §2.2 | **Core investigation** with stratified k-fold CV: baseline, data-prep check, activations, optimisers, regularisation |
| 5 | – | Final evaluation of the CV-selected model on the untouched test set |
| 6 | §2.3 | **Research investigation**: learning-rate warm-up vs. robustness to magnitude pruning |

**What is implemented by hand (on top of PyTorch autograd):** stratified train/test split, stratified k-fold, outlier handling + scaling (fit on training data only), all metrics except the AUC integrals, the training loop (mini-batching, early stopping), the linear learning-rate warm-up schedule, global magnitude pruning with masks, and a Hessian power-iteration for the sharpness measurement in Part 2.

**How to read this notebook**
* Every experiment cell is preceded by a markdown cell that states the **hypothesis** *before* the results appear. Review them and adapt the wording to your own reasoning.
* Cells marked **✏️ Interpretation** are deliberately left for you to fill in once you have seen the numbers. The brief stresses that *interpretation* (not tables or plots) is what earns the insight.
* Figures are saved to `figures/` and tables to `results/` so you can drop them straight into the 2–4 page report.
* Set `QUICK_MODE = True` in the configuration cell for a fast smoke-test (few epochs / folds / seeds). Use `False` for the reported results.

## 0. Setup

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import os, math, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from ucimlrepo import fetch_ucirepo
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_curve, classification_report)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ------------------------------------------------------------------ global configuration
QUICK_MODE = False        # True = tiny smoke-test run; False = the settings used for the report
MASTER_SEED = 42          # controls the data splits, so every experiment sees identical folds
SEEDS = [0, 1, 2, 3, 4]   # seeds for weight init / batch order / dropout in Part 2 (brief: at least 3)
N_FOLDS = 5               # k for stratified k-fold CV on the training pool
TEST_SIZE = 0.20          # fraction of ALL data held out as the final test set
VAL_SIZE = 0.20           # fraction of the training POOL used as validation set  -> 64 / 16 / 20 overall
BATCH_SIZE = 64           # (the original notebook used 32; 64 halves the runtime of the many CV runs)
# A tiny MLP with small batches is usually FASTER on CPU than GPU (kernel-launch overhead dominates).
DEVICE = torch.device("cpu")

if QUICK_MODE:
    N_FOLDS, SEEDS = 3, [0, 1, 2]

def E(n):
    """Number of epochs: full value normally, ~1/5 in QUICK_MODE."""
    return max(2, n // 5) if QUICK_MODE else n

FIG_DIR, RES_DIR = "figures", "results"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), bbox_inches="tight", dpi=150)

def save_table(df, name):
    df.to_csv(os.path.join(RES_DIR, name + ".csv"))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(MASTER_SEED)
print("torch", torch.__version__, "| device:", DEVICE, "| QUICK_MODE:", QUICK_MODE)

## 1. Data

**HTRU2** (UCI id 372): each row is a *pulsar candidate*. The 8 features are simple statistics of two signals that describe the candidate — the **integrated pulse profile** (mean, standard deviation, excess kurtosis, skewness) and the **DM-SNR curve** (the same four statistics). The label is 1 for a genuine pulsar and 0 for radio-frequency interference / noise. Only ≈ 9 % of the rows are positive, so accuracy alone is a misleading metric (a classifier that always says "noise" is ≈ 91 % accurate).

In [ ]:
htru2 = fetch_ucirepo(id=372)
X_df, y_df = htru2.data.features, htru2.data.targets
FEATURE_NAMES = list(X_df.columns)
X_all = X_df.to_numpy(dtype=np.float64)
y_all = y_df.to_numpy().flatten().astype(int)

print(f"Samples: {X_all.shape[0]} | Features: {X_all.shape[1]}")
print("Missing values      :", int(np.isnan(X_all).sum()))
print("Duplicated rows     :", int(X_df.duplicated().sum()))
print(f"Pulsars (class 1)   : {y_all.sum()} ({100 * y_all.mean():.2f} %)")
print(f"Majority-class ('always noise') accuracy: {100 * (1 - y_all.mean()):.2f} %")
display(X_df.describe().T)

### 1.1 Splitting strategy (brief §2.1, item 4)

| Set | Size | Used for |
|---|---|---|
| **Test** | 20 % of all data | touched **once**, at the very end (Section 5) |
| **Training pool** | 80 % | everything else |
| ├─ **k-fold CV** (k = 5) | folds of the pool | comparing hyper-parameters in the core investigation (brief: *"k-fold cross-validation applied to the training set"*) |
| └─ **Train / validation** | 64 % / 16 % of all data | learning curves, early stopping, threshold tuning and the Part 2 experiments |

Design choices and why:
* **Stratified** everywhere: with 9 % positives an unstratified split can easily leave a fold with a very different pulsar ratio, which makes fold-to-fold noise large and comparisons unfair.
* **Fixed seed for the folds**: every configuration is evaluated on exactly the same folds, so differences between configurations are *paired* and not caused by different data.
* **No leakage**: outlier limits, scaling statistics and the class weight are computed from the *training part only* (of each fold) and then applied to the held-out part. Feature-relevance analysis uses the training pool only, never the test set.
* The `my_train_test_split` stub in the original notebook is implemented below as `stratified_split`, and `stratified_kfold_indices` provides the CV folds.

In [ ]:
def stratified_split(y, holdout_frac, seed):
    """Stratified random split of the indices 0..len(y)-1.
    Returns (rest_idx, holdout_idx); holdout_idx holds `holdout_frac` of every class."""
    rng = np.random.default_rng(seed)
    rest, hold = [], []
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        rng.shuffle(idx)
        n_hold = int(round(len(idx) * holdout_frac))
        hold.append(idx[:n_hold]); rest.append(idx[n_hold:])
    rest, hold = np.concatenate(rest), np.concatenate(hold)
    rng.shuffle(rest); rng.shuffle(hold)
    return rest, hold


def stratified_kfold_indices(y, n_splits, seed):
    """Yield (train_idx, val_idx) for stratified k-fold CV (every fold keeps the class ratio)."""
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(n_splits)]
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        rng.shuffle(idx)
        for k, chunk in enumerate(np.array_split(idx, n_splits)):
            folds[k].append(chunk)
    folds = [np.concatenate(f) for f in folds]
    for k in range(n_splits):
        train_idx = np.concatenate([folds[j] for j in range(n_splits) if j != k])
        yield train_idx, folds[k]


# 1) hold out the test set, 2) split the remaining pool into train / validation
pool_idx, test_idx = stratified_split(y_all, TEST_SIZE, seed=MASTER_SEED)
assert len(set(pool_idx) & set(test_idx)) == 0
X_pool, y_pool = X_all[pool_idx], y_all[pool_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

tr_idx, va_idx = stratified_split(y_pool, VAL_SIZE, seed=MASTER_SEED + 1)
assert len(set(tr_idx) & set(va_idx)) == 0
X_train, y_train = X_pool[tr_idx], y_pool[tr_idx]
X_val, y_val = X_pool[va_idx], y_pool[va_idx]

split_tbl = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test), len(y_all)],
    "pulsars": [y_train.sum(), y_val.sum(), y_test.sum(), y_all.sum()],
}, index=["train (64 %)", "validation (16 %)", "test (20 %)", "all"])
split_tbl["pulsar_%"] = (100 * split_tbl["pulsars"] / split_tbl["rows"]).round(2)
display(split_tbl)

## 2. Data preparation

### 2.1 Feature relevance (brief §2.1, item 1)

All analysis in this section uses the **training pool only**. A feature is a candidate for removal if it is

1. **uninformative** – mutual information with the label ≈ 0 *and* a weak single-feature ROC-AUC, or
2. **redundant** – almost perfectly correlated with a feature that carries at least as much information (keep the member of the pair with the higher mutual information).

A candidate is only *actually* removed if a **leave-one-feature-out (LOFO)** cross-validation check with a fast non-linear model (gradient-boosted trees) shows that dropping it does not reduce PR-AUC by more than a small tolerance. This avoids removing a feature that looks redundant marginally but is useful in combination with the others. The same check is applied jointly to all removals together.

*Why PR-AUC?* It is threshold-free and focuses on the minority class, which is what matters here (see §2.5).

In [ ]:
feat_df = pd.DataFrame(X_pool, columns=FEATURE_NAMES)
n_feat = len(FEATURE_NAMES)

# (a) class-conditional distributions (x-range clipped to the 0.5–99.5 percentiles for readability)
ncols = 4; nrows = math.ceil(n_feat / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows))
for ax, j in zip(axes.ravel(), range(n_feat)):
    lo, hi = np.percentile(X_pool[:, j], [0.5, 99.5])
    ax.hist(X_pool[y_pool == 0, j], bins=50, range=(lo, hi), density=True, alpha=0.55, label="Noise")
    ax.hist(X_pool[y_pool == 1, j], bins=50, range=(lo, hi), density=True, alpha=0.55, label="Pulsar")
    ax.set_title(FEATURE_NAMES[j], fontsize=10)
axes.ravel()[0].legend()
plt.suptitle("Class-conditional feature distributions (training pool)", y=1.01)
plt.tight_layout(); savefig("fig_feature_distributions"); plt.show()

# (b) relevance statistics
mi = mutual_info_classif(X_pool, y_pool, random_state=MASTER_SEED)
auc_uni = np.array([roc_auc_score(y_pool, X_pool[:, j]) for j in range(n_feat)])
auc_uni = np.maximum(auc_uni, 1 - auc_uni)          # direction-free single-feature ROC-AUC
rel = pd.DataFrame({"std": X_pool.std(0), "n_unique": [len(np.unique(X_pool[:, j])) for j in range(n_feat)],
                    "mutual_info": mi, "univariate_AUC": auc_uni}, index=FEATURE_NAMES)

# (c) redundancy: correlation matrix
corr = feat_df.corr().to_numpy()
fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm"); ax.grid(False)
ax.set_xticks(range(n_feat)); ax.set_yticks(range(n_feat))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right"); ax.set_yticklabels(FEATURE_NAMES)
for i in range(n_feat):
    for j in range(n_feat):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im); ax.set_title("Pearson correlation (training pool)")
plt.tight_layout(); savefig("fig_correlation"); plt.show()

# (d) leave-one-feature-out check with a fast non-linear model
def cv_pr_auc_trees(cols):
    scores = []
    for tr, va in stratified_kfold_indices(y_pool, N_FOLDS, MASTER_SEED):
        clf = HistGradientBoostingClassifier(max_iter=100, random_state=0)
        clf.fit(X_pool[tr][:, cols], y_pool[tr])
        scores.append(average_precision_score(y_pool[va], clf.predict_proba(X_pool[va][:, cols])[:, 1]))
    return float(np.mean(scores)), float(np.std(scores))

all_cols = list(range(n_feat))
full_mean, full_std = cv_pr_auc_trees(all_cols)
rel["LOFO_drop_PR_AUC"] = [full_mean - cv_pr_auc_trees([c for c in all_cols if c != j])[0] for j in all_cols]

# (e) decision rules
MI_MIN, AUC_MIN = 0.005, 0.55      # "uninformative" thresholds
CORR_MAX = 0.90                    # "redundant" threshold on |Pearson r|
LOFO_TOL = 0.003                   # a removal is accepted if PR-AUC drops by no more than this

cand = set(rel.index[(rel.mutual_info < MI_MIN) & (rel.univariate_AUC < AUC_MIN)])
redundant_pairs = []
for i in range(n_feat):
    for j in range(i + 1, n_feat):
        if abs(corr[i, j]) >= CORR_MAX:
            redundant_pairs.append((FEATURE_NAMES[i], FEATURE_NAMES[j], round(corr[i, j], 3)))
            cand.add(FEATURE_NAMES[i] if mi[i] < mi[j] else FEATURE_NAMES[j])
to_drop = [f for f in cand if rel.loc[f, "LOFO_drop_PR_AUC"] <= LOFO_TOL]

if to_drop:                                        # joint check of all removals together
    keep_cols = [j for j, f in enumerate(FEATURE_NAMES) if f not in to_drop]
    joint_mean, _ = cv_pr_auc_trees(keep_cols)
    if full_mean - joint_mean > LOFO_TOL:
        print(f"Joint removal of {to_drop} costs {full_mean - joint_mean:.4f} PR-AUC -> keeping all features.")
        to_drop = []

DROPPED_FEATURES = sorted(to_drop)
SELECTED_IDX = [j for j, f in enumerate(FEATURE_NAMES) if f not in DROPPED_FEATURES]
rel["candidate"] = [f in cand for f in FEATURE_NAMES]
rel["removed"] = [f in DROPPED_FEATURES for f in FEATURE_NAMES]

print(f"All-feature gradient-boosting CV PR-AUC: {full_mean:.4f} ± {full_std:.4f}")
print("Highly correlated pairs (|r| >= %.2f):" % CORR_MAX, redundant_pairs if redundant_pairs else "none")
display(rel.round(4)); save_table(rel, "feature_relevance")
print("\nREMOVED FEATURES :", DROPPED_FEATURES if DROPPED_FEATURES else "none")
print("FEATURES KEPT    :", [FEATURE_NAMES[j] for j in SELECTED_IDX])

**✏️ Interpretation – feature relevance.** Use the table and figures above to write 3–5 sentences for the report:
* Which features carry the most information about the label (mutual information, single-feature AUC, class-conditional histograms)? Which physical signal (integrated profile vs. DM-SNR curve) is more discriminative?
* Which features are correlated with each other, and why does that make sense given how they are computed (e.g. skewness and kurtosis of the same signal)?
* Did the LOFO check justify removing any feature? If no feature was removed, say so explicitly and give the evidence (LOFO drop ≤ tolerance is *needed* for a removal; a large drop is evidence that the feature matters even when it is correlated with another one).

### 2.2 Outliers (brief §2.1, item 2)

**Identify.** Tukey fences per feature: values outside `[Q1 − k·IQR, Q3 + k·IQR]`, with k = 1.5 (usual outlier) and k = 3 (extreme outlier). Crucially we also compute *how the outliers are distributed over the two classes*.

**Why not simply delete outlier rows?** In HTRU2 the minority (pulsar) class is the one with the most extreme skewness/kurtosis values – the extremes are often *signal, not measurement error*. Deleting rows with extreme values would preferentially delete pulsars, shrink the minority class further and bias the model towards the "typical" noise candidate.

**Strategy: winsorisation (clipping) at the 0.5th / 99.5th percentile of the training data.** All rows are kept, but a handful of extreme values can no longer dominate the loss, the gradients and – importantly – the scaling statistics. Ordering information is preserved up to the clip point, so "very extreme" is still distinguishable from "typical". Clip limits are estimated on the training part of each fold and re-used unchanged on validation / test data.

The alternatives (no handling, dropping rows, robust scaling) are compared empirically in Section 4.2.

In [ ]:
def tukey_fences(X, k=1.5):
    q1, q3 = np.percentile(X, [25, 75], axis=0)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for k in (1.5, 3.0):
    lo, hi = tukey_fences(X_pool, k)
    is_out = (X_pool < lo) | (X_pool > hi)
    tbl = pd.DataFrame({"% outliers (all)": 100 * is_out.mean(0),
                        "% of noise rows": 100 * is_out[y_pool == 0].mean(0),
                        "% of pulsar rows": 100 * is_out[y_pool == 1].mean(0)}, index=FEATURE_NAMES)
    print(f"\nTukey fences with k = {k}")
    display(tbl.round(2)); save_table(tbl, f"outliers_k{k}")
    any_out = is_out.any(axis=1)
    print(f"Rows with >=1 outlying feature: {100 * any_out[y_pool == 0].mean():.1f} % of noise rows vs "
          f"{100 * any_out[y_pool == 1].mean():.1f} % of pulsar rows")
    print(f"-> deleting those rows would remove {100 * (any_out & (y_pool == 1)).sum() / (y_pool == 1).sum():.1f} % of all pulsars "
          f"but only {100 * (any_out & (y_pool == 0)).sum() / (y_pool == 0).sum():.1f} % of the noise rows")

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, j in zip(axes.ravel(), range(n_feat)):
    ax.boxplot([X_pool[y_pool == 0, j], X_pool[y_pool == 1, j]], flierprops=dict(markersize=2, alpha=0.4))
    ax.set_xticks([1, 2]); ax.set_xticklabels(["Noise", "Pulsar"]); ax.set_title(FEATURE_NAMES[j], fontsize=10)
plt.suptitle("Box plots per class (whiskers = 1.5 x IQR; dots = outliers)", y=1.01)
plt.tight_layout(); savefig("fig_outliers_boxplots"); plt.show()

**✏️ Interpretation – outliers.** Which features have the most outliers, and are they concentrated in the pulsar class? Use the "% of pulsar rows vs % of noise rows" table to argue for winsorisation instead of deletion.

### 2.3 Scaling (brief §2.1, item 3)

All 8 features are **continuous, real-valued and on very different scales** (e.g. the profile mean is ≈ 100 while skewness values are ≈ 0–10). Neural networks trained with gradient descent need comparable scales: otherwise the loss surface is badly conditioned (a single learning rate cannot suit all input directions), sigmoid/tanh units saturate on large inputs and weight-initialisation schemes (which assume ≈ unit-variance inputs) lose their meaning.

**Choice: standardisation (z-score) computed *after* winsorising, on the training part only.** After clipping the tails, the mean and standard deviation are no longer distorted by a few extreme values, so z-scoring is well-behaved. Min–max scaling is rejected as a default because it is dominated by the extremes (a single large value squeezes all other points into a tiny interval); robust (median / IQR) scaling is the main alternative and is tested in Section 4.2. Features are not bounded or categorical, so no other encoding is needed.

The pre-processing is wrapped in a `Preprocessor` class with `fit` (training data only) and `transform` so that it can be re-fitted inside every CV fold without leakage.

In [ ]:
class Preprocessor:
    """Feature selection + outlier handling + scaling. Statistics are estimated in `fit` (training data only)."""

    def __init__(self, features, outlier="winsor", winsor_pct=(0.5, 99.5), scaler="standard", drop_k=3.0):
        assert outlier in ("none", "winsor", "drop") and scaler in ("none", "standard", "robust", "minmax")
        self.features, self.outlier, self.winsor_pct = list(features), outlier, tuple(winsor_pct)
        self.scaler, self.drop_k = scaler, drop_k

    def fit(self, X_raw):
        X = X_raw[:, self.features]
        d = X.shape[1]
        self.lo, self.hi = np.full(d, -np.inf), np.full(d, np.inf)             # clip limits
        self.fence_lo, self.fence_hi = np.full(d, -np.inf), np.full(d, np.inf)  # row-drop fences
        if self.outlier == "winsor":
            self.lo, self.hi = np.percentile(X, self.winsor_pct, axis=0)
        elif self.outlier == "drop":
            self.fence_lo, self.fence_hi = tukey_fences(X, self.drop_k)
            X = X[self._inliers(X)]                        # scaling statistics from the inlier rows only
        Xc = np.clip(X, self.lo, self.hi)
        if self.scaler == "standard":
            self.center, self.scale = Xc.mean(0), Xc.std(0)
        elif self.scaler == "robust":
            q1, med, q3 = np.percentile(Xc, [25, 50, 75], axis=0)
            self.center, self.scale = med, q3 - q1
        elif self.scaler == "minmax":
            self.center, self.scale = Xc.min(0), Xc.max(0) - Xc.min(0)
        else:
            self.center, self.scale = np.zeros(d), np.ones(d)
        self.scale = np.where(self.scale < 1e-12, 1.0, self.scale)
        return self

    def _inliers(self, X):
        return ~((X < self.fence_lo) | (X > self.fence_hi)).any(axis=1)

    def training_row_mask(self, X_raw):
        """Rows to keep for TRAINING (only differs from all-True for outlier='drop')."""
        return self._inliers(X_raw[:, self.features])

    def transform(self, X_raw):
        X = np.clip(X_raw[:, self.features], self.lo, self.hi)
        return (X - self.center) / self.scale


def _t(a, dtype=torch.float32):
    return torch.as_tensor(np.asarray(a), dtype=dtype, device=DEVICE)


def prepare_data(X_train_raw, y_train_np, eval_sets, data_cfg):
    """Fit the Preprocessor on the training data ONLY and transform everything.
    eval_sets: {name: (X_raw, y_np)}  ->  out[name] = (X_tensor, y_tensor[n,1], y_numpy)."""
    prep = Preprocessor(**data_cfg).fit(X_train_raw)
    keep = prep.training_row_mask(X_train_raw)
    Xtr, ytr = prep.transform(X_train_raw[keep]), y_train_np[keep]
    out = {"X_train": _t(Xtr), "y_train": _t(ytr).unsqueeze(1), "y_train_np": ytr,
           # class weight for the loss (see 2.5): #negatives / #positives in the *training* data
           "pos_weight": float((ytr == 0).sum() / max((ytr == 1).sum(), 1)),
           "n_dropped": int((~keep).sum()), "prep": prep}
    for name, (Xe, ye) in eval_sets.items():
        out[name] = (_t(prep.transform(Xe)), _t(ye).unsqueeze(1), np.asarray(ye).astype(int))
    return out


# Data configuration used for everything below (justified in 2.1-2.3, checked empirically in 4.2)
DATA_CFG = dict(features=SELECTED_IDX, outlier="winsor", winsor_pct=(0.5, 99.5), scaler="standard")

_d = prepare_data(X_train, y_train, {"val": (X_val, y_val)}, DATA_CFG)
print("Standardised training features -> mean:", _d["X_train"].mean(0).numpy().round(3))
print("                                  std :", _d["X_train"].std(0).numpy().round(3))
print(f"pos_weight (neg/pos) on the training split: {_d['pos_weight']:.2f}")

### 2.4 Splitting – see Section 1.1

### 2.5 Class imbalance (brief §2.1, item 5)

Only ≈ 9 % of the candidates are pulsars. The imbalance is handled at three levels:

| Level | How |
|---|---|
| **Splitting** | *Stratified* train / validation / test split and stratified CV folds → every set has ≈ 9 % positives, so metrics are comparable across folds. Nothing is re-sampled in the validation / test sets: they keep the **real-world** class ratio so that the reported precision/recall are honest. |
| **Training** | **Class-weighted loss**: `BCEWithLogitsLoss(pos_weight = #neg / #pos)` (≈ 10). A missed pulsar costs ≈ 10 × more than a false alarm, which makes the expected gradient contribution of both classes equal. The weight is computed from the training data of each fold. (Over-/under-sampling is an alternative; the weighted loss keeps all information and needs no extra data.) |
| **Evaluation** | **Accuracy is reported but never used for decisions** (the trivial "always noise" model already scores ≈ 91 %). The primary metric is **PR-AUC** (average precision; threshold-free and sensitive to the minority class); we also report precision, recall and **F1 of the pulsar class**, the Matthews correlation coefficient (MCC) and ROC-AUC. Because the class weight shifts the model's probabilities upwards, the decision threshold can additionally be tuned on the *validation* set (Section 5); the default is 0.5. |

In [ ]:
def chance_metrics(y_true):
    """Metrics of a constant 'always noise' classifier - used for diverged (NaN) networks."""
    prev = float(np.mean(y_true))
    return dict(accuracy=1 - prev, precision=0.0, recall=0.0, f1=0.0, mcc=0.0, pr_auc=prev, roc_auc=0.5, diverged=1)


def compute_metrics(y_true, probs, threshold=0.5):
    """Confusion-matrix based metrics implemented by hand; AUC integrals via scikit-learn."""
    y_true = np.asarray(y_true).astype(int)
    pred = (probs >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum()); fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum()); tn = int(((pred == 0) & (y_true == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    denom = math.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denom if denom else 0.0
    return dict(accuracy=(tp + tn) / len(y_true), precision=prec, recall=rec, f1=f1, mcc=mcc,
                pr_auc=float(average_precision_score(y_true, probs)),
                roc_auc=float(roc_auc_score(y_true, probs)), diverged=0)


def sigmoid_np(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


@torch.no_grad()
def predict_logits(model, X):
    model.eval()
    return model(X).squeeze(1).cpu().numpy().astype(np.float64)


def evaluate(model, X, y_np, threshold=0.5):
    z = predict_logits(model, X)
    if not np.all(np.isfinite(z)):                  # network blew up
        return chance_metrics(y_np)
    return compute_metrics(y_np, sigmoid_np(z), threshold)


def tune_threshold(y_true, probs):
    """Decision threshold that maximises F1 of the positive class (to be applied on VALIDATION data only)."""
    p, r, t = precision_recall_curve(y_true, probs)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f1))])

## 3. The MLP and training utilities

* `PulsarMLP` – configurable feed-forward network (hidden sizes, activation, dropout). It outputs a **raw logit**; the sigmoid is folded into `BCEWithLogitsLoss`, which is numerically more stable than `Sigmoid + BCELoss`. (The original notebook applied dropout only after the first hidden layer; here it is applied after every hidden layer so that the depth experiments are consistent.)
* `train_model` – hand-written mini-batch training loop with per-step learning-rate control (needed for warm-up), optional early stopping (restores the best weights), optional pruning masks (re-applied after every optimiser step) and divergence detection.
* `warmup_lr` – linear learning-rate warm-up: `lr(step) = peak_lr · min(1, (step+1) / warmup_steps)`.
* `magnitude_masks` / `apply_masks` – **global magnitude pruning**: the weights (not biases) with the smallest absolute value across *all* layers are removed (masked to zero).
* `run_cv` – stratified k-fold cross-validation on the training pool; pre-processing is re-fitted inside every fold.

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": lambda: nn.LeakyReLU(0.01), "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


class PulsarMLP(nn.Module):
    def __init__(self, input_dim=8, hidden=(32, 16), activation="relu", dropout=0.2):
        super().__init__()
        layers, d = [], input_dim
        for h in hidden:
            layers += [nn.Linear(d, h), ACTIVATIONS[activation]()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))      # regularisation to prevent overfitting
            d = h
        layers.append(nn.Linear(d, 1))                  # 1 output: logit of "is a pulsar"
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)                          # raw logits (no Sigmoid here)


def build_model(model_cfg, input_dim, seed):
    set_seed(seed)                                      # seed controls weight init AND dropout masks
    return PulsarMLP(input_dim=input_dim, **model_cfg).to(DEVICE)


def make_optimizer(name, params, lr, weight_decay=0.0):
    if name == "sgd":          return optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum": return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "rmsprop":      return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    if name == "adam":         return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(name)


def warmup_lr(step, peak_lr, warmup_steps):
    """Linear warm-up from ~0 to peak_lr over `warmup_steps` optimiser steps, constant afterwards."""
    if warmup_steps <= 0:
        return peak_lr
    return peak_lr * min(1.0, (step + 1) / warmup_steps)


# ---------------------------------------------------------------- magnitude pruning helpers
def prunable_weights(model):
    return [m.weight for m in model.modules() if isinstance(m, nn.Linear)]     # weights only, no biases


@torch.no_grad()
def magnitude_masks(model, sparsity):
    """Global magnitude pruning: remove the `sparsity` fraction of weights with the smallest |w| (over all layers)."""
    ws = prunable_weights(model)
    if sparsity <= 0:
        return [torch.ones_like(w) for w in ws]
    all_abs = torch.cat([w.abs().flatten() for w in ws])
    k = max(1, int(round(sparsity * all_abs.numel())))
    threshold = torch.kthvalue(all_abs, k).values
    return [(w.abs() > threshold).float() for w in ws]


@torch.no_grad()
def apply_masks(model, masks):
    for w, m in zip(prunable_weights(model), masks):
        w.mul_(m)


def measure_sparsity(model):
    ws = prunable_weights(model)
    zeros = sum(int((w == 0).sum()) for w in ws)
    return zeros / sum(w.numel() for w in ws)


# ---------------------------------------------------------------- training loop
def train_model(model, Xtr, ytr, Xmon, ymon, pos_weight, cfg, seed=0, masks=None):
    """Mini-batch training with a class-weighted BCE loss.

    cfg keys: optimizer, lr, epochs, [batch_size, weight_decay, warmup_steps, patience]
    Xmon / ymon is the *monitoring* set: it is only logged, unless cfg['patience'] is given, in which case it
    drives early stopping (best weights are restored). Returns history + bookkeeping."""
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
    opt = make_optimizer(cfg["optimizer"], model.parameters(), cfg["lr"], cfg.get("weight_decay", 0.0))
    bs, epochs = cfg.get("batch_size", BATCH_SIZE), cfg["epochs"]
    warm, patience = cfg.get("warmup_steps", 0), cfg.get("patience")
    ymon_np = ymon.squeeze(1).cpu().numpy().astype(int)
    gen = torch.Generator().manual_seed(1000 + seed)              # controls the mini-batch order
    n, step, diverged = Xtr.shape[0], 0, False
    hist = {k: [] for k in ("train_loss", "train_loss_eval", "mon_loss", "mon_pr_auc", "lr")}
    best = {"loss": np.inf, "epoch": 0, "state": None}; bad = 0

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, generator=gen)
        running = 0.0
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            lr_now = warmup_lr(step, cfg["lr"], warm)
            for g in opt.param_groups:
                g["lr"] = lr_now
            opt.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(model(Xtr[idx]), ytr[idx], pos_weight=pw)
            loss_val = loss.item()
            if not math.isfinite(loss_val):
                diverged = True; break
            loss.backward(); opt.step()
            if masks is not None:
                apply_masks(model, masks)                        # keep pruned weights at exactly zero
            running += loss_val * len(idx); step += 1
        if diverged:
            break
        model.eval()
        with torch.no_grad():
            tl = F.binary_cross_entropy_with_logits(model(Xtr), ytr, pos_weight=pw).item()
            zm = model(Xmon)
            ml = F.binary_cross_entropy_with_logits(zm, ymon, pos_weight=pw).item()
        if not (math.isfinite(tl) and math.isfinite(ml)):
            diverged = True; break
        hist["train_loss"].append(running / n)                   # running average, dropout active
        hist["train_loss_eval"].append(tl)                       # eval mode (no dropout): comparable with mon_loss
        hist["mon_loss"].append(ml)
        hist["mon_pr_auc"].append(float(average_precision_score(ymon_np, torch.sigmoid(zm).squeeze(1).cpu().numpy())))
        hist["lr"].append(lr_now)
        if patience:
            if ml < best["loss"] - 1e-4:
                best = {"loss": ml, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}; bad = 0
            else:
                bad += 1
                if bad >= patience:
                    break
    if patience and best["state"] is not None:
        model.load_state_dict(best["state"])
    ml_hist = hist["mon_loss"]
    best_epoch = best["epoch"] if patience else (int(np.argmin(ml_hist)) if ml_hist else 0)
    return dict(history=hist, diverged=diverged, epochs_run=len(ml_hist), best_epoch=best_epoch)


def convergence_epoch(mon_loss, tol=0.05):
    """First epoch (1-indexed) whose monitored loss is within `tol` of its minimum -> convergence speed."""
    if len(mon_loss) == 0:
        return np.nan
    m = min(mon_loss)
    return next(i + 1 for i, v in enumerate(mon_loss) if v <= m * (1 + tol))

In [ ]:
def run_cv(model_cfg, train_cfg, data_cfg=None, label="", n_folds=None, seed=MASTER_SEED, es_frac=0.15, verbose=True):
    """Stratified k-fold CV on the TRAINING POOL. Pre-processing is fitted inside every fold.

    * Fixed-epoch runs: the held-out fold is only *logged* per epoch (learning curves), never used to choose anything.
    * Early-stopping runs (train_cfg['patience']): early stopping uses an inner validation split carved out of the
      training folds, so the held-out fold stays untouched until the final evaluation."""
    data_cfg, n_folds = data_cfg or DATA_CFG, n_folds or N_FOLDS
    rows, hists, t_start = [], [], time.time()
    for k, (tr, va) in enumerate(stratified_kfold_indices(y_pool, n_folds, seed)):
        Xtr_raw, ytr, Xva_raw, yva = X_pool[tr], y_pool[tr], X_pool[va], y_pool[va]
        if train_cfg.get("patience"):
            fit_i, mon_i = stratified_split(ytr, es_frac, seed + 100 + k)
            X_fit, y_fit, X_mon, y_mon = Xtr_raw[fit_i], ytr[fit_i], Xtr_raw[mon_i], ytr[mon_i]
        else:
            X_fit, y_fit, X_mon, y_mon = Xtr_raw, ytr, Xva_raw, yva
        d = prepare_data(X_fit, y_fit, {"mon": (X_mon, y_mon), "hold": (Xva_raw, yva)}, data_cfg)
        model = build_model(model_cfg, d["X_train"].shape[1], seed=k)          # same init for every config (paired)
        out = train_model(model, d["X_train"], d["y_train"], d["mon"][0], d["mon"][1], d["pos_weight"], train_cfg, seed=k)
        m = evaluate(model, d["hold"][0], d["hold"][2])
        h = out["history"]
        m.update(fold=k, epochs_run=out["epochs_run"], best_epoch=out["best_epoch"], diverged=int(out["diverged"]),
                 conv_epoch=convergence_epoch(h["mon_loss"]),
                 train_loss_final=h["train_loss_eval"][-1] if h["mon_loss"] else np.nan,
                 mon_loss_final=h["mon_loss"][-1] if h["mon_loss"] else np.nan,
                 mon_loss_min=min(h["mon_loss"]) if h["mon_loss"] else np.nan,
                 n_dropped_rows=d["n_dropped"])
        rows.append(m); hists.append(h)
    folds = pd.DataFrame(rows)
    res = dict(label=label, folds=folds, hists=hists, model_cfg=model_cfg, train_cfg=train_cfg, data_cfg=data_cfg)
    if verbose:
        print(f"[{label:<34}] PR-AUC {folds.pr_auc.mean():.4f} ± {folds.pr_auc.std():.4f} | "
              f"F1 {folds.f1.mean():.4f} | recall {folds.recall.mean():.3f} | precision {folds.precision.mean():.3f} | "
              f"{time.time() - t_start:.0f}s")
    return res


def summary_table(results, metrics=("pr_auc", "f1", "recall", "precision", "accuracy", "roc_auc")):
    rows = {}
    for lab, r in results.items():
        f = r["folds"]
        row = {m: f"{f[m].mean():.4f} ± {f[m].std():.4f}" for m in metrics}
        row["conv_epoch"] = f"{f['conv_epoch'].mean():.1f}"
        row["epochs_run"] = f"{f['epochs_run'].mean():.1f}"
        row["diverged_folds"] = int(f["diverged"].sum())
        rows[lab] = row
    return pd.DataFrame(rows).T


def mean_curve(hists, key):
    L = max(len(h[key]) for h in hists)
    if L == 0:
        return np.array([]), np.array([])
    A = np.full((len(hists), L), np.nan)
    for i, h in enumerate(hists):
        A[i, :len(h[key])] = h[key]
    return np.nanmean(A, 0), np.nanstd(A, 0)


def plot_cv_comparison(results, title, fname=None):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
    for lab, r in results.items():
        for ax, key in zip(axes[:2], ("mon_loss", "mon_pr_auc")):
            if max(len(h[key]) for h in r["hists"]) == 0:
                continue
            mu, sd = mean_curve(r["hists"], key)
            x = np.arange(1, len(mu) + 1)
            ax.plot(x, mu, label=lab); ax.fill_between(x, mu - sd, mu + sd, alpha=0.15)
    axes[0].set_title("Held-out loss (weighted BCE), mean ± std over folds"); axes[0].set_xlabel("epoch")
    axes[1].set_title("Held-out PR-AUC, mean ± std over folds"); axes[1].set_xlabel("epoch")
    axes[0].legend(fontsize=8)
    labels = list(results)
    axes[2].boxplot([results[l]["folds"]["pr_auc"].values for l in labels])
    axes[2].set_xticks(range(1, len(labels) + 1)); axes[2].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    axes[2].set_title("Final PR-AUC per fold")
    plt.suptitle(title, y=1.02); plt.tight_layout()
    if fname:
        savefig(fname)
    plt.show()

## 4. Core investigation (brief §2.2)

Every comparison below uses **stratified 5-fold cross-validation on the training pool** (identical folds and identical weight initialisation per fold for all configurations, so the comparison is paired). Results are reported as mean ± standard deviation over folds. As a rule of thumb, treat differences smaller than about one fold-standard-deviation as *not clearly distinguishable*.

### 4.1 Baseline

The starting point is the network from the original notebook: 8 → 32 → 16 → 1, ReLU, dropout 0.2, Adam with lr = 0.005, 20 epochs, class-weighted BCE.

In [ ]:
BASE_MODEL = dict(hidden=(32, 16), activation="relu", dropout=0.2)
BASE_TRAIN = dict(optimizer="adam", lr=0.005, epochs=E(20), batch_size=BATCH_SIZE, weight_decay=0.0)

# (a) single run on the train / validation split -> classic learning curves
d = prepare_data(X_train, y_train, {"val": (X_val, y_val)}, DATA_CFG)
model = build_model(BASE_MODEL, d["X_train"].shape[1], seed=0)
out = train_model(model, d["X_train"], d["y_train"], d["val"][0], d["val"][1], d["pos_weight"], BASE_TRAIN, seed=0)
h = out["history"]; ep = np.arange(1, len(h["mon_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, h["train_loss"], label="train (running avg, dropout on)")
axes[0].plot(ep, h["train_loss_eval"], label="train (eval mode)")
axes[0].plot(ep, h["mon_loss"], label="validation")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("weighted BCE loss"); axes[0].set_title("Baseline learning curves"); axes[0].legend()
axes[1].plot(ep, h["mon_pr_auc"], color="C3"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("PR-AUC")
axes[1].set_title("Baseline validation PR-AUC")
plt.tight_layout(); savefig("fig_baseline_learning_curves"); plt.show()
print("Baseline validation metrics (threshold 0.5):",
      {k: round(v, 4) for k, v in evaluate(model, d["val"][0], d["val"][2]).items()})

# (b) 5-fold CV of the baseline (this is the reference for all later comparisons)
baseline_cv = run_cv(BASE_MODEL, BASE_TRAIN, DATA_CFG, label="baseline")
display(summary_table({"baseline": baseline_cv}))

### 4.2 Empirical check of the data-preparation choices

**Hypothesis H0.** (i) Leaving the raw features unscaled hurts (slower / worse training, because the loss is badly conditioned). (ii) *Deleting* outlier rows lowers **recall** (pulsars are over-represented among extreme values) even if precision is unaffected. (iii) Winsorising is at least as good as doing nothing; standardisation, robust scaling and (to a lesser degree) min-max scaling are close to each other on this data.

All variants use the baseline network; only the pre-processing changes (always fitted on the training part of each fold).

In [ ]:
PREP_GRID = {
    "winsorise + standard  (chosen)": dict(outlier="winsor", scaler="standard"),
    "no outlier handling + standard": dict(outlier="none", scaler="standard"),
    "drop outlier rows + standard": dict(outlier="drop", scaler="standard"),
    "winsorise + robust scaler": dict(outlier="winsor", scaler="robust"),
    "winsorise + min-max": dict(outlier="winsor", scaler="minmax"),
    "winsorise + NO scaling": dict(outlier="winsor", scaler="none"),
}
prep_results = {}
for lab, kw in PREP_GRID.items():
    prep_results[lab] = run_cv(BASE_MODEL, BASE_TRAIN, dict(DATA_CFG, **kw), label=lab)

tbl = summary_table(prep_results)
tbl["rows_dropped/fold"] = [f"{prep_results[l]['folds'].n_dropped_rows.mean():.0f}" for l in prep_results]
display(tbl); save_table(tbl, "prep_ablation")
best_prep = max(prep_results, key=lambda l: prep_results[l]["folds"].pr_auc.mean())
print(f"\nBest mean PR-AUC: '{best_prep}'   |   current DATA_CFG: outlier={DATA_CFG['outlier']}, scaler={DATA_CFG['scaler']}")
plot_cv_comparison(prep_results, "Data-preparation ablation (baseline MLP)", "fig_prep_ablation")

**✏️ Interpretation – data preparation.** Was H0 supported? Compare the *recall* column for "drop outlier rows" with the others, the convergence speed of "NO scaling", and check whether the differences between the remaining variants exceed the fold-to-fold standard deviation. If a different configuration is clearly better than the chosen one, update `DATA_CFG` (cell in 2.3) and re-run.

---
### 4.3 Activation functions

**Hypothesis H1.** On this shallow (two hidden layers), well-conditioned (standardised inputs) problem all four activations reach a *similar final PR-AUC* (differences ≲ one fold-std), but they differ in **convergence speed**: sigmoid is slowest (outputs are not zero-centred and its derivative is ≤ 0.25, so gradients are small and weights of the next layer move in correlated directions), tanh is faster (zero-centred, derivative ≤ 1), and ReLU / Leaky-ReLU are fastest (non-saturating, derivative 1 on the active side). Leaky-ReLU ≈ ReLU because dead units are unlikely in such a small network. Only the activation is changed; all else equals the baseline (PyTorch default initialisation).

In [ ]:
act_results = {}
for act in ["sigmoid", "tanh", "relu", "leaky_relu"]:
    act_results[act] = run_cv(dict(BASE_MODEL, activation=act), BASE_TRAIN, DATA_CFG, label=f"activation = {act}")
tbl = summary_table(act_results); display(tbl); save_table(tbl, "core_activations")
plot_cv_comparison(act_results, "Activation functions (5-fold CV)", "fig_activations")

**✏️ Interpretation – activations.** Was H1 supported or contradicted? Look at (1) the final PR-AUC / F1 (are the gaps larger than the fold-std?), (2) the `conv_epoch` column and the shape of the loss curves in the first epochs, (3) the recall/precision trade-off. Link what you see to vanishing gradients, saturation and zero-centred outputs from the lectures.

---
### 4.4 Optimisers

**Hypothesis H2.** *Adaptive* methods (Adam, RMSprop) converge faster and are **less sensitive to the learning rate** over the small-LR range than plain SGD, because their per-parameter step size is normalised by a running estimate of the gradient magnitude. Plain SGD needs a large learning rate and will be *under-trained* after 20 epochs at lr = 10⁻³; momentum (β = 0.9) behaves like ≈ 10× larger effective step and so shifts SGD's sweet spot to smaller learning rates. At the largest learning rate (0.1) the adaptive methods become noisy/unstable (large per-parameter steps) and lose PR-AUC, while SGD is still fine. With each optimiser at its best learning rate the final PR-AUC should be about the same (the problem is easy), i.e. the optimiser mainly affects *speed and stability*, not the attainable accuracy.

Design: 4 optimisers × 3 learning rates {10⁻³, 10⁻², 10⁻¹}, everything else = baseline (20 epochs, batch size 64).

In [ ]:
OPT_NAMES = ["sgd", "sgd_momentum", "rmsprop", "adam"]
OPT_LRS = [1e-3, 1e-2, 1e-1]
opt_results = {}
for o in OPT_NAMES:
    for lr in OPT_LRS:
        opt_results[f"{o} @ {lr:g}"] = run_cv(BASE_MODEL, dict(BASE_TRAIN, optimizer=o, lr=lr), DATA_CFG, label=f"{o} @ lr={lr:g}")

tbl = summary_table(opt_results); display(tbl); save_table(tbl, "core_optimizers")

pr = pd.DataFrame({o: [opt_results[f"{o} @ {lr:g}"]["folds"].pr_auc.mean() for lr in OPT_LRS] for o in OPT_NAMES}, index=OPT_LRS).T
cv_ = pd.DataFrame({o: [opt_results[f"{o} @ {lr:g}"]["folds"].conv_epoch.mean() for lr in OPT_LRS] for o in OPT_NAMES}, index=OPT_LRS).T

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for ax, M, ttl, cmap in ((axes[0], pr, "Mean CV PR-AUC", "viridis"), (axes[1], cv_, "Epochs to converge (lower = faster)", "viridis_r")):
    im = ax.imshow(M.values, cmap=cmap, aspect="auto"); ax.grid(False)
    ax.set_xticks(range(len(OPT_LRS))); ax.set_xticklabels([f"{lr:g}" for lr in OPT_LRS]); ax.set_xlabel("learning rate")
    ax.set_yticks(range(len(OPT_NAMES))); ax.set_yticklabels(OPT_NAMES); ax.set_title(ttl)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M.values[i, j]:.3f}" if M is pr else f"{M.values[i, j]:.1f}", ha="center", va="center", color="w", fontsize=9)
    plt.colorbar(im, ax=ax)
best_lr = pr.idxmax(axis=1)
for o in OPT_NAMES:
    mu, sd = mean_curve(opt_results[f"{o} @ {best_lr[o]:g}"]["hists"], "mon_loss")
    axes[2].plot(np.arange(1, len(mu) + 1), mu, label=f"{o} @ {best_lr[o]:g}")
axes[2].set_title("Held-out loss at each optimiser's best lr"); axes[2].set_xlabel("epoch"); axes[2].legend(fontsize=8)
plt.tight_layout(); savefig("fig_optimizers"); plt.show()
print("Best learning rate per optimiser (by mean PR-AUC):", best_lr.to_dict())

**✏️ Interpretation – optimisers.** Was H2 supported? Discuss (1) the heat-map: for which (optimiser, lr) cells does training fail or lag, and how wide is each optimiser's "good" learning-rate band? (2) the convergence-epoch map and the loss curves at the best learning rates; (3) whether the *best* PR-AUC differs between optimisers by more than the fold-std. Relate momentum and adaptive step sizes to what was covered in the lectures.

---
### 4.5 Regularisation and overfitting behaviour

**Hypothesis H3.** HTRU2 is low-dimensional (8 features) with ≈ 11 000 training rows per fold, so the *small* baseline network hardly overfits. To observe overfitting we use an **over-parameterised** network (8 → 128 → 64 → 1, ≈ 9 000 weights) trained for 50 epochs without regularisation. We expect its training loss to keep falling while the held-out loss reaches a minimum and then *rises* (a growing generalisation gap). Dropout and weight decay should shrink that gap; early stopping should find nearly the same held-out loss in far fewer epochs. However, since the task is easy, the *improvement in final PR-AUC* from any regulariser is expected to be small (≲ one fold-std) – regularisation mainly changes the gap and the training cost, not the achievable test performance here.

Early stopping uses an inner validation split taken from the training folds (patience = 5 epochs on the weighted validation loss; best weights restored), so the held-out fold is never used to pick the stopping epoch.

In [ ]:
BIG_MODEL = dict(hidden=(128, 64), activation="relu", dropout=0.0)
REG_TRAIN = dict(optimizer="adam", lr=1e-3, epochs=E(50), batch_size=BATCH_SIZE, weight_decay=0.0)
REG_CFGS = {
    "none":                   (BIG_MODEL, REG_TRAIN),
    "dropout 0.3":            (dict(BIG_MODEL, dropout=0.3), REG_TRAIN),
    "weight decay 1e-3":      (BIG_MODEL, dict(REG_TRAIN, weight_decay=1e-3)),
    "early stopping (p=5)":   (BIG_MODEL, dict(REG_TRAIN, patience=5)),
    "dropout + wd + ES":      (dict(BIG_MODEL, dropout=0.3), dict(REG_TRAIN, weight_decay=1e-3, patience=5)),
}
reg_results = {lab: run_cv(mc, tc, DATA_CFG, label=f"reg: {lab}") for lab, (mc, tc) in REG_CFGS.items()}

tbl = summary_table(reg_results)
tbl["gap (val-train loss)"] = [f"{(r['folds'].mon_loss_final - r['folds'].train_loss_final).mean():.4f}" for r in reg_results.values()]
tbl["val loss rise after min"] = [f"{(r['folds'].mon_loss_final - r['folds'].mon_loss_min).mean():.4f}" for r in reg_results.values()]
display(tbl); save_table(tbl, "core_regularisation")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for lab, r in reg_results.items():
    if r["train_cfg"].get("patience"):
        continue                                    # early-stopping runs monitor an inner split, so their curves are not comparable
    mu_v, _ = mean_curve(r["hists"], "mon_loss"); mu_t, _ = mean_curve(r["hists"], "train_loss_eval")
    x = np.arange(1, len(mu_v) + 1)
    axes[0].plot(x, mu_t, label=lab); axes[1].plot(x, mu_v, label=lab); axes[2].plot(x, mu_v - mu_t, label=lab)
axes[0].set_title("Training loss (eval mode)"); axes[1].set_title("Held-out loss"); axes[2].set_title("Generalisation gap = held-out − train")
for a in axes: a.set_xlabel("epoch")
axes[1].legend(fontsize=8)
plt.tight_layout(); savefig("fig_regularisation_curves"); plt.show()
plot_cv_comparison(reg_results, "Regularisation (5-fold CV)", "fig_regularisation")

**✏️ Interpretation – regularisation.** Was H3 supported? (1) In the un-regularised run, does the held-out loss rise after its minimum while the training loss keeps falling (look at the curves and the "val loss rise after min" column)? (2) How much do dropout / weight decay reduce the gap, and do they change PR-AUC by more than the fold-std? (3) How many epochs does early stopping save? Connect this to bias–variance and the capacity of the network relative to the amount of data.

### 4.6 Summary of the core investigation and model selection

All configurations from 4.1–4.5 were evaluated on the same folds, so they can be ranked by mean cross-validation PR-AUC. The best one is carried forward to the test set (Section 5). Because the differences between the top configurations are often within the fold-std, the ranking should be read together with the interpretation cells above; a simpler/cheaper configuration within one std of the best is also a defensible choice (override `SELECTED_LABEL` if you prefer that).

In [ ]:
ALL_CORE = {"baseline": baseline_cv}
ALL_CORE.update({f"activation: {k}": v for k, v in act_results.items()})
ALL_CORE.update({f"optimizer: {k}": v for k, v in opt_results.items()})
ALL_CORE.update({f"regularisation: {k}": v for k, v in reg_results.items()})

core_rank = pd.DataFrame({
    lab: dict(PR_AUC=r["folds"].pr_auc.mean(), PR_AUC_std=r["folds"].pr_auc.std(),
              F1=r["folds"].f1.mean(), recall=r["folds"].recall.mean(), precision=r["folds"].precision.mean(),
              accuracy=r["folds"].accuracy.mean()) for lab, r in ALL_CORE.items()
}).T.sort_values("PR_AUC", ascending=False)
display(core_rank.round(4).head(10)); save_table(core_rank, "core_ranking")

SELECTED_LABEL = core_rank.index[0]          # <- override with any key of ALL_CORE if you prefer another configuration
print("Selected configuration:", SELECTED_LABEL)
print("  model:", ALL_CORE[SELECTED_LABEL]["model_cfg"]); print("  train:", ALL_CORE[SELECTED_LABEL]["train_cfg"])

## 5. Final evaluation on the held-out test set

The test set has not been used for any decision so far. The selected configuration (and, for comparison, the baseline) is retrained on the 64 % **train** split – early stopping, if selected, monitors the 16 % **validation** split – for each seed in `SEEDS`, and evaluated on the test set:

* at the default threshold 0.5, and
* at the **F1-optimal threshold tuned on the validation split** (the class-weighted loss shifts the predicted probabilities upwards, so a different threshold is often better for precision).

In [ ]:
def final_test_eval(model_cfg, train_cfg, seeds):
    d = prepare_data(X_train, y_train, {"val": (X_val, y_val), "test": (X_test, y_test)}, DATA_CFG)
    rows, keep = [], None
    for s in seeds:
        model = build_model(model_cfg, d["X_train"].shape[1], seed=s)
        train_model(model, d["X_train"], d["y_train"], d["val"][0], d["val"][1], d["pos_weight"], train_cfg, seed=s)
        p_val = sigmoid_np(predict_logits(model, d["val"][0])); p_test = sigmoid_np(predict_logits(model, d["test"][0]))
        thr = tune_threshold(d["val"][2], p_val)
        for name, t in (("0.5", 0.5), ("val-tuned", thr)):
            rows.append(dict(threshold=name, thr_value=t, seed=s, **compute_metrics(d["test"][2], p_test, t)))
        if keep is None:
            keep = dict(p_test=p_test, thr=thr, y=d["test"][2])
    return pd.DataFrame(rows), keep

sel = ALL_CORE[SELECTED_LABEL]
final_sel, keep_sel = final_test_eval(sel["model_cfg"], sel["train_cfg"], SEEDS)
final_base, _ = final_test_eval(BASE_MODEL, BASE_TRAIN, SEEDS)

cols = ["accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc", "mcc"]
def fmt(df):
    return df.groupby("threshold")[cols].agg(lambda s: f"{s.mean():.4f} ± {s.std():.4f}")
print(f"TEST RESULTS – selected configuration ({SELECTED_LABEL}), mean ± std over {len(SEEDS)} seeds"); display(fmt(final_sel))
print("TEST RESULTS – baseline"); display(fmt(final_base))
save_table(fmt(final_sel), "final_test_selected"); save_table(fmt(final_base), "final_test_baseline")

# detailed report for the first seed (same format as the original notebook)
for name, t in (("threshold 0.5", 0.5), (f"validation-tuned threshold {keep_sel['thr']:.3f}", keep_sel["thr"])):
    print(f"\nClassification report – {name} (seed {SEEDS[0]})")
    print(classification_report(keep_sel["y"], (keep_sel["p_test"] >= t).astype(int), target_names=["Noise", "Pulsar"], digits=3))

# precision-recall curve + confusion matrix
p, r, _ = precision_recall_curve(keep_sel["y"], keep_sel["p_test"])
pred = (keep_sel["p_test"] >= keep_sel["thr"]).astype(int)
cm = np.array([[((pred == a) & (keep_sel["y"] == b)).sum() for a in (0, 1)] for b in (0, 1)])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(r, p); axes[0].axhline(keep_sel["y"].mean(), ls="--", color="gray", label="chance (= prevalence)")
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision"); axes[0].set_title("Test precision-recall curve"); axes[0].legend()
axes[1].imshow(cm, cmap="Blues"); axes[1].grid(False)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["pred Noise", "pred Pulsar"]); axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["true Noise", "true Pulsar"])
axes[1].set_title("Test confusion matrix (validation-tuned threshold)")
plt.tight_layout(); savefig("fig_test_pr_confusion"); plt.show()

## 6. Research investigation: learning-rate warm-up and robustness to pruning (brief §2.3)

### Background and the claim under test
Frankle & Carbin [1] and Renda et al. [2] observed that networks trained with **learning-rate warm-up** keep their accuracy much better under (iterative) magnitude pruning than networks trained with a constant learning rate, even when both have the same unpruned accuracy. The proposed mechanism has **two links**:

1. warm-up lets a network *tolerate a larger target learning rate* [3] (it steers training into flatter regions where a large step is still stable), and
2. a larger learning rate helps a network *recover from pruning* [2].

Together: *warm-up → larger usable LR → better pruning robustness*. This is a causal hypothesis, so the experiment is designed to test each link separately instead of only checking "warm-up vs. no warm-up".

### Experimental design
* **Network / optimiser.** MLP 8 → 128 → 64 → 32 → 1 (ReLU, no dropout), ≈ 11 000 prunable weights, **SGD with momentum 0.9** (the setting where the classical stability picture "λ_max ≲ 2(1+β)/η" of [3], with η the learning rate, β the momentum and λ_max the top Hessian eigenvalue, is cleanest), class-weighted BCE. Everything about pre-processing is as in Section 2.
* **Warm-up.** Linear warm-up from ≈ 0 to the peak learning rate over the first 5 epochs, then *constant*. The no-warm-up network uses the same constant peak learning rate from step 0, so the warm-up phase is the **only** difference between the schedules.
* **Step 1 – learning-rate tolerance sweep (tests link 1).** Train both schedules over a wide grid of peak learning rates (3 seeds each) and find the largest LR at which each still trains successfully (validation PR-AUC within 0.01 of the best and no divergence).
* **Step 2 – 2 × 2 factorial pruning experiment (tests link 2 and separates warm-up from LR).** Conditions: {no warm-up, warm-up} × {`lr_low` = largest LR that *no-warm-up* tolerates, `lr_high` = largest LR that *warm-up* tolerates}. The **primary comparison** is *no warm-up @ lr_low* vs. *warm-up @ lr_high* (what the literature compares); the two remaining cells are controls (if `lr_high == lr_low` the control cells are identical to the primary ones and are skipped). The primary pair is only interpreted if its **unpruned accuracy is comparable** (checked explicitly).
* **Pruning.** One-shot **global magnitude pruning** (weights only) of the *trained* network at sparsities 30 / 50 / 70 / 90 / 95 / 98 %. The same procedure and levels for every condition. Because HTRU2 is an easy problem, the network is heavily over-parameterised and moderate sparsities are expected to be harmless for *all* conditions – the high sparsities are where differences can appear.
* **Fine-tuning.** Identical for every condition: 15 epochs, constant learning rate 0.01 (same optimiser), pruning masks kept fixed. We report the metrics **right after pruning** (robustness) and **after fine-tuning** (recovered performance).
* **Seeds and statistics.** 5 seeds (init, batch order); mean ± std reported, and *paired* per-seed differences (same seed ⇒ same initial weights in every condition).
* **Metrics.** Accuracy (requested by the brief) *and* F1 / PR-AUC, because accuracy is nearly uninformative under 9 % prevalence. The learning rate is *selected* on the validation split; the plots report the **test split**.
* **Sharpness.** At the end of training we estimate the top Hessian eigenvalue λ_max of the training loss (power iteration with Hessian-vector products) as direct evidence for the "flatter region" part of the mechanism.

### Hypotheses (stated before the results)
* **H-A (link 1).** Warm-up tolerates a larger peak LR than no warm-up: `lr_high > lr_low`. Without warm-up the largest LRs diverge or collapse early (large early gradients while the weights are random), with warm-up they train.
* **H-B (main effect).** With comparable unpruned accuracy, *warm-up @ lr_high* retains higher F1/PR-AUC than *no warm-up @ lr_low* right after pruning and after fine-tuning, and the gap **grows with sparsity**.
* **H-C (mechanism).** The advantage is carried by the *learning rate*, not by warm-up as such: at the *same* LR, warm-up and no warm-up prune equally well (when the LR is small enough for both to train), and *no warm-up @ lr_high* – if it trains at all – is as robust as *warm-up @ lr_high*.
* **H-D (curvature).** Larger-LR training ends at smaller λ_max (edge-of-stability: λ_max is capped at ≈ 2(1+β)/η), so *warm-up @ lr_high* ends at lower sharpness than *no warm-up @ lr_low*.

Because HTRU2 is small and easy, **a null result is a legitimate outcome** (the literature effect was measured on much harder problems); if that is what you find, say so and explain why (e.g. the loss surface of a tiny MLP on standardised inputs is benign at any reasonable LR, and 98 % sparsity may still leave enough weights for such an easy task).

In [ ]:
# ------------------------------------------------------------------ Part 2 configuration
P2_MODEL = dict(hidden=(128, 64, 32), activation="relu", dropout=0.0)
P2_OPT = "sgd_momentum"                       # momentum 0.9
P2_BS = 64
P2_EPOCHS = E(40)
P2_WARMUP_EPOCHS = 2 if QUICK_MODE else 5
P2_LR_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
P2_SPARSITIES = [0.3, 0.5, 0.7, 0.9, 0.95, 0.98]
FT_EPOCHS, FT_LR = E(15), 0.01                # identical fine-tuning for every condition
STABLE_TOL = 0.01                             # "trains successfully" = val PR-AUC within this of the best
MATCH_TOL = 0.01                              # "comparable unpruned accuracy" tolerance (PR-AUC)
SWEEP_SEEDS = SEEDS[:3]
P2_LR_LOW_OVERRIDE, P2_LR_HIGH_OVERRIDE = None, None    # set numbers here to override the automatic choice

# data for Part 2: fixed train / validation / test split, pre-processing fitted on the train split only
P2 = prepare_data(X_train, y_train, {"val": (X_val, y_val), "test": (X_test, y_test)}, DATA_CFG)
P2_STEPS_PER_EPOCH = math.ceil(P2["X_train"].shape[0] / P2_BS)
_m = PulsarMLP(input_dim=P2["X_train"].shape[1], **P2_MODEL)
print(f"Prunable weights: {sum(w.numel() for w in prunable_weights(_m))} | steps/epoch: {P2_STEPS_PER_EPOCH} | "
      f"warm-up steps: {P2_WARMUP_EPOCHS * P2_STEPS_PER_EPOCH}")


def p2_train(seed, lr, warmup_epochs, epochs=None):
    model = build_model(P2_MODEL, P2["X_train"].shape[1], seed=seed)
    cfg = dict(optimizer=P2_OPT, lr=lr, epochs=epochs or P2_EPOCHS, batch_size=P2_BS,
               warmup_steps=warmup_epochs * P2_STEPS_PER_EPOCH)
    out = train_model(model, P2["X_train"], P2["y_train"], P2["val"][0], P2["val"][1], P2["pos_weight"], cfg, seed=seed)
    return model, out


def top_hessian_eigenvalue(model, X, y, pos_weight, n_iter=30, max_samples=4096, seed=0):
    """Largest Hessian eigenvalue of the (class-weighted) training loss via power iteration with
    Hessian-vector products (double back-prop). Dropout is off (eval mode)."""
    model.eval()
    params = [p for p in model.parameters() if p.requires_grad]
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g)[:max_samples]
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
    loss = F.binary_cross_entropy_with_logits(model(X[idx]), y[idx], pos_weight=pw)
    grads = torch.autograd.grad(loss, params, create_graph=True)
    v = [torch.randn(p.shape, generator=g).to(DEVICE) for p in params]
    norm = math.sqrt(sum((x ** 2).sum().item() for x in v)); v = [x / norm for x in v]
    eig = float("nan")
    for _ in range(n_iter):
        Hv = torch.autograd.grad(grads, params, grad_outputs=v, retain_graph=True)
        eig = sum((h * x).sum().item() for h, x in zip(Hv, v))         # Rayleigh quotient v^T H v
        norm = math.sqrt(sum((h ** 2).sum().item() for h in Hv))
        if norm == 0:
            return 0.0
        v = [h / norm for h in Hv]
    return eig


def run_prune_experiment(name, lr, warmup_epochs, seed, sparsities):
    """Train one network, then prune it at every sparsity and fine-tune with the SAME procedure.
    Returns long-format records (condition, seed, sparsity, stage in {pre_ft, post_ft}, split in {val, test}, metrics)."""
    model, out = p2_train(seed, lr, warmup_epochs)
    recs = []

    def add(sp, actual, stage, net):
        for split in ("val", "test"):
            met = chance_metrics(P2[split][2]) if out["diverged"] else evaluate(net, P2[split][0], P2[split][2])
            recs.append(dict(condition=name, seed=seed, sparsity=sp, actual_sparsity=actual, stage=stage, split=split, **met))

    add(0.0, measure_sparsity(model), "pre_ft", model); add(0.0, measure_sparsity(model), "post_ft", model)
    for sp in sparsities:
        if out["diverged"]:
            add(sp, sp, "pre_ft", model); add(sp, sp, "post_ft", model); continue
        pm = copy.deepcopy(model)
        masks = magnitude_masks(pm, sp); apply_masks(pm, masks)
        add(sp, measure_sparsity(pm), "pre_ft", pm)                                 # robustness: right after pruning
        ft_cfg = dict(optimizer=P2_OPT, lr=FT_LR, epochs=FT_EPOCHS, batch_size=P2_BS)   # identical for all conditions
        train_model(pm, P2["X_train"], P2["y_train"], P2["val"][0], P2["val"][1], P2["pos_weight"], ft_cfg, seed=seed, masks=masks)
        add(sp, measure_sparsity(pm), "post_ft", pm)                                # recovered performance
    sharp = float("nan") if out["diverged"] else top_hessian_eigenvalue(model, P2["X_train"], P2["y_train"], P2["pos_weight"], seed=seed)
    info = dict(condition=name, seed=seed, lr=lr, warmup_epochs=warmup_epochs, diverged=int(out["diverged"]),
                sharpness=sharp, history=out["history"])
    return recs, info


# quick sanity check of the pruning machinery
_mm = build_model(P2_MODEL, P2["X_train"].shape[1], seed=0)
for sp in (0.5, 0.9):
    _mk = magnitude_masks(_mm, sp); _pm = copy.deepcopy(_mm); apply_masks(_pm, _mk)
    print(f"target sparsity {sp:.2f} -> measured {measure_sparsity(_pm):.4f}")

### 6.1 Step 1 – learning-rate tolerance sweep (tests H-A)

Train the network with **no warm-up** and with **linear warm-up** over a wide grid of peak learning rates (SGD + momentum), 3 seeds per point, and evaluate on the validation split. A run that produces a non-finite loss is counted as *diverged* (and scored as a chance-level classifier). The *largest tolerated LR* of a schedule is the largest grid value whose mean validation PR-AUC is within 0.01 of the best PR-AUC found anywhere in the sweep, with no diverged seed.

In [ ]:
sweep_rows, t0 = [], time.time()
for sched, w in (("no warmup", 0), ("warmup", P2_WARMUP_EPOCHS)):
    for lr in P2_LR_GRID:
        for s in SWEEP_SEEDS:
            model, out = p2_train(s, lr, w)
            m = chance_metrics(P2["val"][2]) if out["diverged"] else evaluate(model, P2["val"][0], P2["val"][2])
            sweep_rows.append(dict(schedule=sched, lr=lr, seed=s, diverged=int(out["diverged"]),
                                   val_pr_auc=m["pr_auc"], val_f1=m["f1"], val_accuracy=m["accuracy"],
                                   final_train_loss=out["history"]["train_loss_eval"][-1] if out["history"]["train_loss_eval"] else np.nan))
    print(f"{sched}: sweep done ({time.time() - t0:.0f}s)")

sw = pd.DataFrame(sweep_rows)
agg = sw.groupby(["schedule", "lr"]).agg(pr_auc=("val_pr_auc", "mean"), pr_std=("val_pr_auc", "std"), f1=("val_f1", "mean"),
                                         acc=("val_accuracy", "mean"), n_diverged=("diverged", "sum")).reset_index()
display(agg.pivot(index="lr", columns="schedule", values=["pr_auc", "n_diverged"]).round(4)); save_table(agg, "p2_lr_sweep")

fig, ax = plt.subplots(figsize=(7, 4.3))
for sched, g in agg.groupby("schedule"):
    ax.errorbar(g.lr, g.pr_auc, yerr=g.pr_std.fillna(0), marker="o", capsize=3, label=sched)
    dv = g[g.n_diverged > 0]
    ax.scatter(dv.lr, dv.pr_auc, marker="x", s=130, c="k", zorder=5, label="diverged (>= 1 seed)" if sched == "no warmup" else None)
ax.set_xscale("log"); ax.set_xlabel("peak learning rate"); ax.set_ylabel("validation PR-AUC (mean ± std, 3 seeds)")
ax.set_title("Learning-rate tolerance: warm-up vs. no warm-up"); ax.legend()
plt.tight_layout(); savefig("fig_p2_lr_sweep"); plt.show()

best_overall = agg.pr_auc.max()
def max_tolerated_lr(sched):
    ok = agg[(agg.schedule == sched) & (agg.pr_auc >= best_overall - STABLE_TOL) & (agg.n_diverged == 0)]
    if len(ok):
        return float(ok.lr.max())
    g = agg[agg.schedule == sched]
    return float(g.loc[g.pr_auc.idxmax(), "lr"])          # fallback: the best LR of that schedule

LR_NW_MAX, LR_W_MAX = max_tolerated_lr("no warmup"), max_tolerated_lr("warmup")
print(f"Best validation PR-AUC over the whole sweep: {best_overall:.4f}")
print(f"Largest tolerated LR  -> no warm-up: {LR_NW_MAX:g} | warm-up: {LR_W_MAX:g}")
print("H-A (warm-up tolerates a larger LR):", "SUPPORTED on this grid" if LR_W_MAX > LR_NW_MAX else "NOT supported on this grid (same tolerance)")

**✏️ Interpretation – LR tolerance (H-A).** Describe the two curves: at which LR does each schedule start to degrade or diverge, and is the warm-up curve shifted to the right? Is the *gain* in tolerated LR small (one grid step) or large? What happens to the no-warm-up runs at large LR – NaN divergence, or a silent collapse to a poor solution (dead ReLUs / saturated logits)? Explain it with the stability condition λ_max ≲ 2(1+β)/η (η = learning rate, β = momentum, λ_max = top Hessian eigenvalue) discussed in [3]: at initialisation the curvature of the loss can be large, so a large step taken *immediately* overshoots, whereas warm-up gives the network time to move towards a region with smaller curvature before the LR reaches its target.

### 6.2 Step 2 – choose the conditions and run the pruning experiment
`lr_low` = largest LR that *no warm-up* tolerates, `lr_high` = largest LR that *warm-up* tolerates (both from the sweep; override with `P2_LR_LOW_OVERRIDE` / `P2_LR_HIGH_OVERRIDE` if you want different values).

In [ ]:
LR_LOW = P2_LR_LOW_OVERRIDE or LR_NW_MAX
LR_HIGH = P2_LR_HIGH_OVERRIDE or max(LR_W_MAX, LR_NW_MAX)
nm = lambda warm, lr: f"{'warmup' if warm else 'no warmup'}, lr={lr:g}"

C_NW_LOW, C_W_HIGH = nm(False, LR_LOW), nm(True, LR_HIGH)                 # primary pair
CONDITIONS = {C_NW_LOW: (LR_LOW, 0), C_W_HIGH: (LR_HIGH, P2_WARMUP_EPOCHS)}
C_W_LOW = C_NW_HIGH = None
if LR_HIGH != LR_LOW:                                                      # controls of the 2x2 design
    C_W_LOW, C_NW_HIGH = nm(True, LR_LOW), nm(False, LR_HIGH)
    CONDITIONS[C_W_LOW] = (LR_LOW, P2_WARMUP_EPOCHS)
    CONDITIONS[C_NW_HIGH] = (LR_HIGH, 0)
else:
    print("lr_high == lr_low: warm-up did not raise the tolerated LR on this grid -> only the primary pair is run.")
for k, (lr, w) in CONDITIONS.items():
    print(f"{k:<28} peak lr = {lr:g} | warm-up epochs = {w}")

In [ ]:
p2_records, p2_info = [], []
for name, (lr, w) in CONDITIONS.items():
    t0 = time.time()
    for s in SEEDS:
        recs, info = run_prune_experiment(name, lr, w, s, P2_SPARSITIES)
        p2_records += recs; p2_info.append(info)
    print(f"finished: {name:<28} ({len(SEEDS)} seeds, {time.time() - t0:.0f}s)")

res = pd.DataFrame(p2_records)
info_df = pd.DataFrame([{k: v for k, v in i.items() if k != "history"} for i in p2_info])
save_table(res, "p2_results_long"); save_table(info_df, "p2_run_info")
print("max |actual - target| sparsity:", float((res.actual_sparsity - res.sparsity).abs().max()))

### 6.3 Results

First check the **matching requirement** (brief §2.3.2, item 1): the two models being compared must have comparable unpruned performance, otherwise any difference after pruning could simply reflect a different starting point.

In [ ]:
unp = res[(res.sparsity == 0) & (res.stage == "post_ft")]
tbl = unp.groupby(["condition", "split"], sort=False)[["accuracy", "f1", "pr_auc"]].agg(lambda s: f"{s.mean():.4f} ± {s.std():.4f}")
print("UNPRUNED performance (mean ± std over seeds)"); display(tbl); save_table(tbl, "p2_unpruned")

print("Matching check for the primary pair (|Δ mean PR-AUC| <= %.3f):" % MATCH_TOL)
for split in ("val", "test"):
    d = unp[unp.split == split]
    a, b = d[d.condition == C_NW_LOW].pr_auc, d[d.condition == C_W_HIGH].pr_auc
    ok = abs(a.mean() - b.mean()) <= MATCH_TOL
    print(f"  {split:<5}: {C_NW_LOW} = {a.mean():.4f} ± {a.std():.4f} | {C_W_HIGH} = {b.mean():.4f} ± {b.std():.4f} "
          f"| Δ = {b.mean() - a.mean():+.4f} -> {'COMPARABLE' if ok else 'NOT comparable – interpret the pruning comparison with care'}")

# training curves: LR schedule and validation loss per condition (mean over seeds)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for cond in CONDITIONS:
    hs = [i["history"] for i in p2_info if i["condition"] == cond]
    for ax, key in zip(axes, ("lr", "mon_loss", "mon_pr_auc")):
        mu, sd = mean_curve(hs, key)
        if len(mu):
            x = np.arange(1, len(mu) + 1); ax.plot(x, mu, label=cond); ax.fill_between(x, mu - sd, mu + sd, alpha=0.15)
axes[0].set_title("Learning rate at the end of each epoch"); axes[1].set_title("Validation loss"); axes[2].set_title("Validation PR-AUC")
axes[1].set_yscale("log")
for a in axes: a.set_xlabel("epoch")
axes[0].legend(fontsize=8); plt.tight_layout(); savefig("fig_p2_training_curves"); plt.show()

In [ ]:
def plot_sparsity(df, stage, split, title, fname, metrics=("accuracy", "f1", "pr_auc")):
    d = df[(df.stage == stage) & (df.split == split)]
    levels = sorted(d.sparsity.unique())
    fig, axes = plt.subplots(1, len(metrics), figsize=(5.6 * len(metrics), 4.2))
    for ax, m in zip(np.atleast_1d(axes), metrics):
        for i, (cond, g) in enumerate(d.groupby("condition", sort=False)):
            a = g.groupby("sparsity")[m].agg(["mean", "std"]).reindex(levels)
            ax.errorbar(np.arange(len(levels)), a["mean"], yerr=a["std"].fillna(0), marker="os^v"[i % 4], capsize=3,
                        lw=2.2 if cond in (C_NW_LOW, C_W_HIGH) else 1.2, ls="-" if cond in (C_NW_LOW, C_W_HIGH) else "--", label=cond)
        ax.set_xticks(range(len(levels))); ax.set_xticklabels([f"{100 * s:g}%" for s in levels]); ax.set_xlabel("sparsity"); ax.set_ylabel(m)
    np.atleast_1d(axes)[0].legend(fontsize=8)
    plt.suptitle(title, y=1.02); plt.tight_layout(); savefig(fname); plt.show()

plot_sparsity(res, "post_ft", "test", "After pruning + fine-tuning (test split, mean ± std over seeds)", "fig_p2_sparsity_post_ft")
plot_sparsity(res, "pre_ft", "test", "Immediately after pruning, before fine-tuning (test split)", "fig_p2_sparsity_pre_ft")
plot_sparsity(res, "post_ft", "val", "After pruning + fine-tuning (validation split)", "fig_p2_sparsity_post_ft_val")

summ = res[res.split == "test"].groupby(["stage", "condition", "sparsity"], sort=False)[["accuracy", "f1", "pr_auc"]].agg(lambda s: f"{s.mean():.4f} ± {s.std():.4f}")
display(summ.unstack("sparsity")["pr_auc"]); save_table(summ, "p2_sparsity_summary")

In [ ]:
def paired_diff(df, a, b, metric="pr_auc", split="test", stage="post_ft"):
    """Per-seed paired difference (a - b) at every sparsity. Same seed => same initial weights in both conditions."""
    d = df[(df.split == split) & (df.stage == stage)]
    pa = d[d.condition == a].pivot(index="seed", columns="sparsity", values=metric)
    pb = d[d.condition == b].pivot(index="seed", columns="sparsity", values=metric)
    diff = pa - pb
    return pd.DataFrame({"mean_diff": diff.mean(), "std_diff": diff.std(), "seeds_a_better": (diff > 0).sum(), "n_seeds": diff.count()}).T.round(4)

COMPARISONS = [("PRIMARY: warm-up@lr_high  minus  no-warm-up@lr_low", C_W_HIGH, C_NW_LOW)]
if C_W_LOW:
    COMPARISONS += [("warm-up effect at the SAME low LR", C_W_LOW, C_NW_LOW),
                    ("warm-up effect at the SAME high LR", C_W_HIGH, C_NW_HIGH),
                    ("LR effect without warm-up (high - low)", C_NW_HIGH, C_NW_LOW),
                    ("LR effect with warm-up (high - low)", C_W_HIGH, C_W_LOW)]
for stage in ("post_ft", "pre_ft"):
    for metric in ("pr_auc", "accuracy"):
        print(f"\n===== {metric.upper()} | test split | stage = {stage}  (positive mean_diff => first condition is better) =====")
        for title, a, b in COMPARISONS:
            if stage == "pre_ft" and title != COMPARISONS[0][0]:
                continue
            print(f"-- {title}"); display(paired_diff(res, a, b, metric=metric, stage=stage))

# sharpness (H-D)
sh = info_df.groupby("condition", sort=False).agg(peak_lr=("lr", "first"), warmup_epochs=("warmup_epochs", "first"), diverged_seeds=("diverged", "sum"),
                                                  lambda_max_mean=("sharpness", "mean"), lambda_max_std=("sharpness", "std"))
if P2_OPT == "sgd_momentum":
    sh["stability_limit_2(1+beta)/lr"] = 2 * (1 + 0.9) / sh.peak_lr
print("\nTop Hessian eigenvalue of the training loss at the end of training (mean over seeds)")
display(sh.round(3)); save_table(sh, "p2_sharpness")

**✏️ Interpretation – warm-up and pruning robustness.** Answer each hypothesis explicitly, quoting the numbers above and comparing every difference to the seed-to-seed std:

1. **Matching.** Were the unpruned models comparable? If not, what does that imply for the comparison?
2. **H-B (main effect).** Does *warm-up @ lr_high* beat *no warm-up @ lr_low* right after pruning and after fine-tuning? At which sparsity does the gap appear, and does it grow with sparsity? How many of the 5 seeds agree in sign (`seeds_a_better`)? Remember that at moderate sparsity a heavily over-parameterised network on an easy task may lose nothing at all.
3. **H-C (mechanism).** Compare the controls: (a) warm-up vs. no warm-up at the *same* LR, (b) the effect of LR within each schedule. If the advantage tracks the *learning rate* rather than the schedule, that supports the proposed causal chain; if warm-up helps even at equal LR (or LR has no effect), the chain is not supported and another mechanism (or none) is at work.
4. **H-D (curvature).** Is λ_max lower for the large-LR / warm-up runs? How close is it to the stability limit 2(1+β)/η (edge of stability)?
5. **Honest conclusion.** Did you observe the effect from the literature? If not, give reasons (task difficulty, network size, one-shot pruning, fine-tune vs. rewinding, warm-up length, seeds) and what experiment would disambiguate them. Report contradicting evidence as prominently as supporting evidence.

---
### 6.4 Optional extension – warm-up duration (brief §2.3.3)

**Hypothesis H-E.** If warm-up helps because it lets the network reach a flatter region before the full LR is applied, a *longer* warm-up should help up to a point (enough time to leave the high-curvature region at initialisation) and then saturate; a very short warm-up (≈ 1 epoch) should behave almost like no warm-up. If the effect is binary or absent, the curve will be flat. The peak LR is `lr_high` for all durations; we look at the highest sparsities where differences are most likely.

In [ ]:
RUN_EXT_DURATION = True
EXT_DURATIONS = [0, 2] if QUICK_MODE else [0, 1, 2, 5, 10]        # warm-up length in epochs (0 = no warm-up)
EXT_SPARSITIES = [0.9, 0.98]

if RUN_EXT_DURATION:
    ext_records, t0 = [], time.time()
    for wd in EXT_DURATIONS:
        for s in SEEDS:
            recs, _ = run_prune_experiment(f"warmup {wd} ep", LR_HIGH, wd, s, EXT_SPARSITIES)
            ext_records += recs
        print(f"warm-up = {wd:>2} epochs done ({time.time() - t0:.0f}s)")
    ext = pd.DataFrame(ext_records); save_table(ext, "p2_ext_warmup_duration")
    ext["warmup_epochs"] = ext.condition.str.extract(r"warmup (\d+) ep").astype(int)
    d = ext[(ext.stage == "post_ft") & (ext.split == "test")]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ("pr_auc", "accuracy")):
        for sp in [0.0] + EXT_SPARSITIES:
            a = d[d.sparsity == sp].groupby("warmup_epochs")[metric].agg(["mean", "std"])
            ax.errorbar(a.index, a["mean"], yerr=a["std"].fillna(0), marker="o", capsize=3, label=f"sparsity {100 * sp:g}%")
        ax.set_xlabel("warm-up duration (epochs)"); ax.set_ylabel(f"test {metric} (after fine-tuning)")
    axes[0].legend(); plt.suptitle(f"Warm-up duration (peak lr = {LR_HIGH:g})", y=1.02)
    plt.tight_layout(); savefig("fig_p2_ext_warmup_duration"); plt.show()
    display(d.groupby(["sparsity", "warmup_epochs"])[["accuracy", "f1", "pr_auc"]].agg(lambda s: f"{s.mean():.4f} ± {s.std():.4f}"))

**✏️ Interpretation – warm-up duration (optional).** Is there a duration beyond which the effect saturates? Is the effect binary? Compare with the seed-to-seed std.

---
## 7. Checklist for the written report (2–4 pages)

| Brief requirement | Where in this notebook | Figure / table files |
|---|---|---|
| Feature relevance | §2.1 | `fig_feature_distributions`, `fig_correlation`, `feature_relevance.csv` |
| Outliers | §2.2 | `fig_outliers_boxplots`, `outliers_k*.csv` |
| Scaling | §2.3 + §4.2 | `prep_ablation.csv`, `fig_prep_ablation` |
| Splitting | §1.1 | split table |
| Imbalance | §2.5 + §4 (metrics) + §5 | `final_test_*.csv` |
| Baseline + learning curves | §4.1 | `fig_baseline_learning_curves` |
| ≥ 2 core axes, hypothesis → result → interpretation | §4.3 activations, §4.4 optimisers, §4.5 regularisation | `core_*.csv`, `fig_activations`, `fig_optimizers`, `fig_regularisation*` |
| Warm-up vs. no warm-up, ≥ 3 seeds, mean ± std | §6.2 – 6.3 | `p2_results_long.csv` |
| Comparable unpruned accuracy | §6.3 (matching check) | `p2_unpruned.csv` |
| Same pruning + fine-tuning for both | `run_prune_experiment` | – |
| **Accuracy vs. sparsity plot for both conditions** | §6.3 | `fig_p2_sparsity_post_ft`, `fig_p2_sparsity_pre_ft` |
| Interpretation of support / contradiction of the LR-tolerance mechanism | §6.1 – 6.3 (✏️ cells) | `fig_p2_lr_sweep`, `p2_sharpness.csv` |
| Optional: warm-up duration | §6.4 | `fig_p2_ext_warmup_duration` |

Optional extensions *not* implemented here: timing of pruning relative to warm-up, distillation, NAS.

**References.** [1] Frankle & Carbin, *The Lottery Ticket Hypothesis*, ICLR 2019. [2] Renda, Frankle & Carbin, *Comparing Rewinding and Fine-tuning in Neural Network Pruning*, ICLR 2020. [3] Kalra & Barkeshli, *Why Warmup the Learning Rate? Underlying Mechanisms and Improvements*, NeurIPS 2024.